In [6]:
# 1. Install the Gurobi library
%pip install gurobipy

import gurobipy as gp
from gurobipy import *

# 2. Input your license credentials
params = {
    "WLSACCESSID": 'a9386248-233e-42c3-9f08-56b178e9f9e2',
    "WLSSECRET": 'a94f121b-7d29-44d4-ba07-aeea9c9954f1',
    "LICENSEID": 2769033,
}

# 3. Create the Gurobi environment
env = gp.Env(params=params)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2769033
Academic license 2769033 - for non-commercial use only - registered to pa___@smu.ca


In [7]:
!pip -q install pulp openpyxl
import pandas as pd
import numpy as np
import pulp
from google.colab import files


In [8]:
excel_file = "/content/Params (1).xlsx"

A = pd.read_excel(excel_file, sheet_name="A-Matrix", header=None).to_numpy(dtype=float)
b = pd.read_excel(excel_file, sheet_name="b-Vector", header=None).to_numpy(dtype=float).flatten()
c = pd.read_excel(excel_file, sheet_name="c-Vector", header=None).to_numpy(dtype=float).flatten()

print(A.shape, len(b), len(c))


(20, 10) 20 10


In [11]:
def print_solution(model, tol=1e-8):
    print("Positive decision variables:")
    for v in model.variables():
        if v.value() is not None and v.value() > tol:
            print(v.name, "=", v.value())
    print("Objective value =", pulp.value(model.objective))


In [12]:
def Max_Vector(A, b, c):
    model = pulp.LpProblem("Dual", pulp.LpMaximize)

    m, n = A.shape

    # dual variables pi0..pi(m-1)
    pi = [pulp.LpVariable(f"pi{i}", lowBound=0) for i in range(m)]

    # objective: max b^T pi
    model += pulp.lpSum(b[i] * pi[i] for i in range(m))

    # constraints: A^T pi <= c
    for j in range(n):
        model += pulp.lpSum(A[i, j] * pi[i] for i in range(m)) <= c[j]

    # export LP file
    model.writeLP("Dual.lp")
    return model


In [13]:
dual_model = Max_Vector(A, b, c)
status = dual_model.solve(pulp.PULP_CBC_CMD(msg=False))
print("Dual status:", pulp.LpStatus[status])

print("=== DUAL SOLUTION ===")
print_solution(dual_model)

dual_obj = pulp.value(dual_model.objective)


Dual status: Optimal
=== DUAL SOLUTION ===
Positive decision variables:
pi11 = 1.552795
pi18 = 8.4409938
pi8 = 7.8012422
Objective value = 5379.335385599999


In [14]:
def Min_Vector(A, b, c):
    model = pulp.LpProblem("Primal", pulp.LpMinimize)
    m, n = A.shape

    x = [pulp.LpVariable(f"x{j}", lowBound=0) for j in range(n)]
    model += pulp.lpSum(c[j] * x[j] for j in range(n))

    for i in range(m):
        model += pulp.lpSum(A[i, j] * x[j] for j in range(n)) >= b[i]

    return model

primal_check = Min_Vector(A, b, c)
primal_check.solve(pulp.PULP_CBC_CMD(msg=False))
primal_obj = pulp.value(primal_check.objective)

print("\n=== OBJECTIVE CHECK ===")
print("Primal objective:", primal_obj)
print("Dual objective:  ", dual_obj)
print("Difference:", abs(primal_obj - dual_obj))



=== OBJECTIVE CHECK ===
Primal objective: 5379.335404
Dual objective:   5379.335385599999
Difference: 1.8400000953988638e-05


In [15]:
files.download("Dual.lp")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>